# Statistical Analysis

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from scipy import stats
from scipy.stats import ttest_ind, f_oneway, spearmanr

import statsmodels.api as sm
from statsmodels.stats.weightstats import DescrStatsW

import matplotlib.pyplot as plt
import seaborn as sns

### load the clean file

In [2]:
df = pd.read_csv(
    "black_friday_cleaned.csv"
)

df.head()

,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase,Product_Category_2_Num,Product_Category_3_Num,Outlier_Flag,Marital_Status_Label
0,1000001,P00000142,F,0-17,10,A,2,0,3,4,5,13650,4.0,5.0,Normal,Single
1,1000001,P00004842,F,0-17,10,A,2,0,3,4,12,13645,4.0,12.0,Normal,Single
2,1000001,P00025442,F,0-17,10,A,2,0,1,2,9,15416,2.0,9.0,Normal,Single
3,1000001,P00051442,F,0-17,10,A,2,0,8,17,Unknown,9938,17.0,NaN,Normal,Single
4,1000001,P00051842,F,0-17,10,A,2,0,4,8,Unknown,2849,8.0,NaN,Normal,Single


In [4]:
## all data are here:
df.shape

(550068, 16)

### Statistical Significance Standard

### Hypothesis 1 — Gender

#### Step 1 — Create Groups

In [5]:
male = df.loc[
    df["Gender"] == "M",
    "Purchase"
]

female = df.loc[
    df["Gender"] == "F",
    "Purchase"
]

In [6]:
# check average
print("Male Average:", male.mean())
print("Female Average:", female.mean())

Male Average: 9437.526040472265
Female Average: 8734.565765155476


#### run test

In [7]:
t_stat, p_value = ttest_ind(
    male,
    female,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)


T-statistic: 46.358248669626064
P-value: 0.0


#### Interpretation

In [8]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0: Significant difference.")
else:
    print("Fail to reject H0: No significant difference.")


Reject H0: Significant difference.


### Hypothesis 2 — Marital Status

In [9]:
# Create Groups
single = df.loc[
    df["Marital_Status"] == 0,
    "Purchase"
]

married = df.loc[
    df["Marital_Status"] == 1,
    "Purchase"
]

In [10]:
# mean comparison
print("Single Average:", single.mean())
print("Married Average:", married.mean())

Single Average: 9265.907618921507
Married Average: 9261.174574082374


In [11]:
# Welch's t-test
t_stat, p_value = ttest_ind(
    single,
    married,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 0.34379905124865934
P-value: 0.7309975627344574


In [12]:
# Decision
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

Fail to reject H0


###  Hypothesis 3 — Age Groups


In [13]:
# create group
age_groups = [
    group["Purchase"].values
    for _, group in df.groupby("Age")
]

In [14]:
# run ANOVA
f_stat, p_value = f_oneway(*age_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 40.57579909450407
P-value: 1.053563939251671e-49


In [15]:
# decision
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


### Hypothesis 4 — City Category

In [16]:
# steps are as follows-
city_groups = [
    group["Purchase"].values
    for _, group in df.groupby("City_Category")
]

In [17]:
f_stat, p_value = f_oneway(*city_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 1130.7459610102978
P-value: 0.0


In [18]:
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


### If ANOVA Is Significant — Tukey Test

In [19]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [20]:
# Age
tukey_age = pairwise_tukeyhsd(
    endog=df["Purchase"],
    groups=df["Age"],
    alpha=0.05
)

print(tukey_age)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
group1 group2  meandiff p-adj    lower    upper   reject
--------------------------------------------------------
  0-17  18-25   236.199    0.0  106.9071 365.4908   True
  0-17  26-35   319.226    0.0  194.6669 443.7851   True
  0-17  36-45  397.8861    0.0  269.3973 526.3748   True
  0-17  46-50  275.1611    0.0  136.1875 414.1346   True
  0-17  51-55  601.3434    0.0  459.1789 743.5079   True
  0-17    55+  402.8158    0.0  245.6171 560.0145   True
 18-25  26-35    83.027 0.0003   26.4748 139.5792   True
 18-25  36-45  161.6871    0.0   96.9373 226.4369   True
 18-25  46-50   38.9621 0.8162  -44.6849 122.6091  False
 18-25  51-55  365.1444    0.0  276.2968 453.9921   True
 18-25    55+  166.6169 0.0002   55.2858 277.9479   True
 26-35  36-45   78.6601 0.0004   23.9688 133.3513   True
 26-35  46-50  -44.0649 0.6116 -120.1926  32.0627  False
 26-35  51-55  282.1174    0.0  200.3097 363.9251   True
 26-35    55+   83.5898 0.2298 

In [21]:
# city
tukey_city = pairwise_tukeyhsd(
    endog=df["Purchase"],
    groups=df["City_Category"],
    alpha=0.05
)

print(tukey_city)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj  lower    upper   reject
-----------------------------------------------------
     A      B 239.3613   0.0 200.2276  278.495   True
     A      C 807.9818   0.0 766.2598 849.7038   True
     B      C 568.6204   0.0 531.1583 606.0826   True
-----------------------------------------------------


### City Tenure vs Purchase

In [23]:
df["Stay_Years_Num"] = (
    df["Stay_In_Current_City_Years"]
    .astype(str)
    .replace("4+", "4")
    .astype(int)
)

In [24]:
# Then use Spearman correlation, which is more appropriate for an ordinal variable:
correlation, p_value = spearmanr(
    df["Stay_Years_Num"],
    df["Purchase"]
)

print("Spearman Correlation:", correlation)
print("P-value:", p_value)

Spearman Correlation: 0.005944147880930418
P-value: 1.0404122836138882e-05


###  Interpret Correlation

### Confidence Interval for Mean Purchase
#### The project asks us to interpret confidence intervals as well as p-values. For overall average purchase:

In [25]:
mean_purchase = df["Purchase"].mean()
sem = stats.sem(df["Purchase"])

confidence_interval = stats.t.interval(
    confidence=0.95,
    df=len(df["Purchase"]) - 1,
    loc=mean_purchase,
    scale=sem
)

print("Mean Purchase:", mean_purchase)
print("95% CI:", confidence_interval)

Mean Purchase: 9263.968712959126
95% CI: (np.float64(9250.694472258305), np.float64(9277.242953659947))


In [ ]:
## Interpretation:

# The 95% confidence interval provides a range of plausible values for the population mean purchase amount.

### Confidence Interval — Gender

In [26]:
def mean_ci(series, confidence=0.95):

    mean = series.mean()
    sem = stats.sem(series)

    ci = stats.t.interval(
        confidence=confidence,
        df=len(series) - 1,
        loc=mean,
        scale=sem
    )

    return mean, ci

In [27]:
male_mean, male_ci = mean_ci(male)
female_mean, female_ci = mean_ci(female)

print("Male:", male_mean, male_ci)
print("Female:", female_mean, female_ci)

Male: 9437.526040472265 (np.float64(9422.019402055814), np.float64(9453.032678888716))
Female: 8734.565765155476 (np.float64(8709.21132117373), np.float64(8759.92020913722))


### Very Important — Statistical vs Business Significance

### Assumptions & Limitations